# Chaveta del engrane C — cálculo paso a paso
Este cuaderno hace **solo** la chaveta, en el mismo orden que lo haces a mano. Cada celda corresponde a un paso.

**Cómo usarlo**
1. Cambia números **solo en la celda 0** (las líneas marcadas `← CAMBIA`).
2. Ejecuta las celdas en orden (Shift+Enter).
3. Cada paso imprime las fórmulas con sus números, para que compares con tu cálculo a mano.

| Paso | Qué calcula | Fuente |
|---|---|---|
| 1 | Torque en C y fuerza en la chaveta | tu cálculo de potencias |
| 2 | Chaveta a **fluencia** (corte y aplastamiento) | PPT Chavetas, diap. 11 |
| 3 | Chaveta a **fatiga** | PPT Fatiga, diap. 13–15, 17, 27–28 |
| 4 | **Eje** a fatiga en el chavetero | PPT Chavetas, diap. 3 (K_f) |
| 5 | Resumen | |


In [57]:
# ======================= 0. DATOS (aquí es lo ÚNICO que cambias) =======================
# --- Motor y engranes (enunciado) ---
P_HP   = 10        # [HP]   potencia del motor
rpm    = 120       # [rpm]  velocidad del motor
d_B    = 60        # [mm]   diámetro primitivo del piñón B
d_C    = 120       # [mm]   diámetro primitivo del engrane C

# --- Eje (enunciado + tu ficha técnica del SAE 1045) ---
d_eje  = 40        # [mm]   diámetro del eje
Sr_eje = 640       # [MPa]  resistencia a la ruptura del eje
S0_eje = 340       # [MPa]  resistencia a la fluencia del eje
BHN_eje = 185      # [-]    dureza aproximada (≈ Sr/3,45). Si es < 200 BHN usa la fila "< 200" de la tabla de K_f

# --- Esfuerzos en el eje en la sección de C (de TUS diagramas de momento flector y torsor) ---
M_flector_C = 3312   # [N·m]  momento flector resultante en C  (diagrama de M_res)
T_torsor_C  = 1187.3 # [N·m]  torsor en C (valor absoluto)

# --- Chaveta (ANSI para Ø40: 12 x 8) ---
b = 12             # [mm]   ancho                  ← CAMBIA
h = 8              # [mm]   alto                   ← CAMBIA
L = 60            # [mm]   largo (≤ ancho del cubo de C)   ← CAMBIA

# --- Material de la chaveta  ⚠ VALORES DE PRUEBA: reemplaza por los de tu ficha técnica ---
nombre_chaveta = "ASTM 1020 (prueba)"   # ← CAMBIA
S0_ch = 300        # [MPa]  fluencia de la chaveta   ← CAMBIA
Sr_ch = 450        # [MPa]  ruptura de la chaveta    ← CAMBIA
S0_cubo = 340      # [MPa]  fluencia del cubo (engrane C). ⚠ SUPUESTO: no sabemos su material   ← CAMBIA

# --- Criterios ---
N = 1.6            # factor de seguridad a fatiga (Tabla 1, grupo 1); aquí también se usa para fluencia de la chaveta
Ci = 1.0           # confiabilidad: 50 % → 1,0 | 90 % → 0,897 | 95 % → 0,868 | 99 % → 0,814   ← DECIDE
fresa = "dedo"     # tipo de fresa del chavetero: "dedo" o "disco"   ← DECIDE
Kf_chaveta = 1.0   # concentrador de la chaveta misma (el PPT no da uno; se supone 1 y se justifica)


## Paso 1 · Torque en C y fuerza en la chaveta
A mano: `M_B = P/ω_B` ; `T_C = M_B · (d_C/d_B)` ; `F = T_C / (d/2)`  (diap. 11: M_t = (D/2)·F)

In [51]:
import numpy as np
P = P_HP*746                        # [W]
w_B = rpm*2*np.pi/60                # [rad/s]
M_B = P/w_B                         # [N·m]
T_C = M_B*d_C/d_B                   # [N·m]  (potencia constante: el par sube al bajar la velocidad)
F = T_C*1000/(d_eje/2)              # [N]    fuerza sobre la chaveta
print(f"P = {P:.0f} W | ω_B = {w_B:.3f} rad/s | M_B = {M_B:.1f} N·m")
print(f"T_C = M_B·d_C/d_B = {T_C:.1f} N·m")
print(f"F = T_C/(d/2) = {F:.0f} N")

P = 7460 W | ω_B = 12.566 rad/s | M_B = 593.6 N·m
T_C = M_B·d_C/d_B = 1187.3 N·m
F = T_C/(d/2) = 59365 N


## Paso 2 · Chaveta a fluencia (corte y aplastamiento)
A mano: `τ = F/(b·L)` ; `σ = 2F/(L·h)` ; `τ_adm = 0,577·S₀/N` ; `σ_adm = S₀/N`
En el aplastamiento la cara de la chaveta se apoya contra el eje **y** contra el cubo: manda el material más débil de los tres.

In [52]:
S0_aplast = min(S0_ch, S0_eje, S0_cubo)             # [MPa] el más débil de chaveta, eje y cubo
tau   = F/(b*L)                                      # [MPa] corte en la chaveta
sigma = 2*F/(L*h)                                    # [MPa] aplastamiento
tau_adm   = 0.577*S0_ch/N                            # [MPa]
sigma_adm = S0_aplast/N                              # [MPa]
N_corte_est   = 0.577*S0_ch/tau                      # [-]
N_aplast_est  = S0_aplast/sigma                      # [-]
L_min = max(F/(b*tau_adm), 2*F/(h*sigma_adm))        # [mm] largo mínimo para cumplir N
ok = lambda x: "CUMPLE" if x else "NO CUMPLE"
print(f"CORTE:      τ = {tau:.1f} MPa  vs  τ_adm = {tau_adm:.1f} MPa  → N = {N_corte_est:.2f}  → {ok(tau<=tau_adm)}")
print(f"APLASTAM.:  σ = {sigma:.1f} MPa  vs  σ_adm = {sigma_adm:.1f} MPa  → N = {N_aplast_est:.2f}  → {ok(sigma<=sigma_adm)}   (material que manda: S₀ = {S0_aplast} MPa)")
print(f"Largo mínimo para N = {N}:  L_min = {L_min:.1f} mm   (tú usas L = {L} mm)")

CORTE:      τ = 83.1 MPa  vs  τ_adm = 108.2 MPa  → N = 2.08  → CUMPLE
APLASTAM.:  σ = 249.4 MPa  vs  σ_adm = 187.5 MPa  → N = 1.20  → NO CUMPLE   (material que manda: S₀ = 300 MPa)
Largo mínimo para N = 1.6:  L_min = 79.2 mm   (tú usas L = 59.5 mm)


## Paso 3 · Chaveta a fatiga (la pieza rectangular)
El momento va de 0 a su máximo, así que la fuerza va de 0 a F: **esfuerzo medio = esfuerzo alterno = máximo/2**.

A mano:
- Límite de fatiga: `S_f' = 0,5·S_r` ; `S_f = S_f'·C_c·C_t·C_s·C_T·C_i`
  - `C_c = 1` (carga combinada) ; `C_T = 1` ; `C_i` = confiabilidad
  - `C_t = 0,879·d_e^(−0,107)` con `d_e = 0,808·√(h·b)` en **pulgadas** (sección rectangular, diap. 15)
  - `C_s = 4,51·S_r^(−0,265)` (mecanizado, diap. 17), con el S_r **de la chaveta**
- **Corte** (Von Mises: σ' = √3·τ y Goodman): `σ'_a/S_f + σ'_m/S_r = 1/N`
- **Aplastamiento** es compresión (σ_m ≤ 0): basta `σ_a/S_f = 1/N`  (diap. 28)

In [53]:
Sf_prima = 0.5*Sr_ch                                         # [MPa]
Cc, CT = 1.0, 1.0
d_e = 0.808*np.sqrt(h*b)                                     # [mm] diámetro equivalente
Ct = 0.879*(d_e/25.4)**-0.107
Cs = 4.51*Sr_ch**-0.265
Sf = Sf_prima*Cc*Ct*Cs*CT*Ci                                 # [MPa]
print(f"S_f' = {Sf_prima:.1f} | d_e = {d_e:.2f} mm | C_t = {Ct:.3f} | C_s = {Cs:.3f} | C_i = {Ci} → S_f = {Sf:.1f} MPa")

# CORTE: ciclo 0 → τ
tau_a = tau_m = tau/2                                        # [MPa]
sa_eq = np.sqrt(3)*Kf_chaveta*tau_a                          # [MPa] σ'_a (K_f solo en la parte alterna)
sm_eq = np.sqrt(3)*tau_m                                     # [MPa] σ'_m
N_corte_fat = 1/(sa_eq/Sf + sm_eq/Sr_ch)
print(f"CORTE:     τ_a = τ_m = {tau_a:.1f} MPa → σ'_a = σ'_m = {sa_eq:.1f} MPa → N = {N_corte_fat:.2f} → {ok(N_corte_fat>=N)}")

# APLASTAMIENTO: ciclo 0 → −σ (compresión)
sig_a = Kf_chaveta*sigma/2                                   # [MPa]
N_aplast_fat = Sf/sig_a
print(f"APLASTAM.: σ_a = {sig_a:.1f} MPa (σ_m = −{sigma/2:.1f} ≤ 0) → N = {N_aplast_fat:.2f} → {ok(N_aplast_fat>=N)}")

S_f' = 225.0 | d_e = 7.92 mm | C_t = 0.996 | C_s = 0.893 | C_i = 1.0 → S_f = 200.2 MPa
CORTE:     τ_a = τ_m = 41.6 MPa → σ'_a = σ'_m = 72.0 MPa → N = 1.92 → CUMPLE
APLASTAM.: σ_a = 124.7 MPa (σ_m = −124.7 ≤ 0) → N = 1.61 → CUMPLE


## Paso 4 · El eje a fatiga en el chavetero
Aquí falla el **eje**, no la chaveta. El chavetero es una entalla: aparece K_f (PPT Chavetas, diap. 3).
Cargas en el eje: flexión que se invierte en cada vuelta (σ_a = σ_máx) y torsión que va de 0 a T (τ_a = τ_m = τ_máx/2).

A mano: `σ_máx = 32·M/(π·d³)` ; `τ_máx = 16·T/(π·d³)` ;
`σ'_a = √[(K_ff·σ_máx)² + 3·(K_ft·τ_a)²]` ; `σ'_m = √3·τ_m` ; `σ'_a/S_f + σ'_m/S_r = 1/N`
(K_f solo multiplica la parte **alterna**.)

In [58]:
# Tabla K_f del PPT Chavetas diap. 3: (flexión, torsión)
tabla_Kf = {("disco","<200"): (1.3, 1.3), ("disco",">=200"): (1.6, 1.6),
            ("dedo","<200"):  (1.6, 1.3), ("dedo",">=200"):  (2.0, 1.6)}
fila = "<200" if BHN_eje < 200 else ">=200"
Kff, Kft = tabla_Kf[(fresa, fila)]
print(f"Fresa de {fresa}, dureza {fila} BHN → K_ff (flexión) = {Kff} | K_ft (torsión) = {Kft}")

# Límite de fatiga corregido del eje
Sf_eje = 0.5*Sr_eje*1.0*(0.879*(d_eje/25.4)**-0.107)*(4.51*Sr_eje**-0.265)*1.0*Ci   # [MPa]

# Esfuerzos MÁXIMOS (flexión y torsión)
sig_max = 32*M_flector_C*1000/(np.pi*d_eje**3)               # [MPa] flexión
tau_max = 16*T_torsor_C*1000/(np.pi*d_eje**3)                # [MPa] torsión

# Ciclo 0 → máximo en AMBAS cargas: alterno = medio = máximo/2
sig_a = sig_m = sig_max/2                                    # [MPa] flexión
tau_a = tau_m = tau_max/2                                    # [MPa] torsión

# Von Mises (diap. 28): K_f solo en las componentes ALTERNAS
sa_e = np.sqrt((Kff*sig_a)**2 + 3*(Kft*tau_a)**2)            # [MPa] σ'_a
sm_e = np.sqrt(sig_m**2 + 3*tau_m**2)                        # [MPa] σ'_m  (σ_m > 0 → se usa Goodman)

N_eje = 1/(sa_e/Sf_eje + sm_e/Sr_eje)                        # Goodman: σ'_a/S_f + σ'_m/S_r = 1/N
print(f"S_f(eje) = {Sf_eje:.1f} MPa | σ_máx = {sig_max:.1f} MPa | τ_máx = {tau_max:.1f} MPa")
print(f"σ_a = σ_m = {sig_a:.1f} MPa | τ_a = τ_m = {tau_a:.1f} MPa")
print(f"σ'_a = {sa_e:.1f} MPa | σ'_m = {sm_e:.1f} MPa → N_eje = {N_eje:.2f} → {ok(N_eje>=N)}")

Fresa de dedo, dureza <200 BHN → K_ff (flexión) = 1.6 | K_ft (torsión) = 1.3
S_f(eje) = 218.1 MPa | σ_máx = 527.1 MPa | τ_máx = 94.5 MPa
σ_a = σ_m = 263.6 MPa | τ_a = τ_m = 47.2 MPa
σ'_a = 434.9 MPa | σ'_m = 276.0 MPa → N_eje = 0.41 → NO CUMPLE


## Paso 5 · Resumen

In [56]:
filas = [("Chaveta · corte · fluencia", N_corte_est), ("Chaveta · aplastamiento · fluencia", N_aplast_est),
         ("Chaveta · corte · fatiga", N_corte_fat), ("Chaveta · aplastamiento · fatiga", N_aplast_fat),
         ("EJE en el chavetero · fatiga", N_eje)]
print(f"Chaveta {b}×{h}×{L} mm, material: {nombre_chaveta} | requisito N = {N}\n")
for nom, n in filas:
    print(f"{nom:36s} N = {n:5.2f}   {'✔' if n >= N else '✘'}")
print(f"\nL_min para fatiga = {L:.1f} mm.  Material mínimo para este L:  S₀ ≥ N·σ = {N*sigma:.0f} MPa (aplastamiento).")

Chaveta 12×8×59.5 mm, material: ASTM 1020 (prueba) | requisito N = 1.6

Chaveta · corte · fluencia           N =  2.08   ✔
Chaveta · aplastamiento · fluencia   N =  1.20   ✘
Chaveta · corte · fatiga             N =  1.92   ✔
Chaveta · aplastamiento · fatiga     N =  1.61   ✔
EJE en el chavetero · fatiga         N =  0.25   ✘

L_min para fatiga = 59.5 mm.  Material mínimo para este L:  S₀ ≥ N·σ = 399 MPa (aplastamiento).
